# Chapter 10: Embeddings in Depth

Embeddings are the foundation of modern NLP. They map discrete text (words, sentences, documents) into a continuous vector space where **similar meanings are close together**.

```
'king' - 'man' + 'woman' ≈ 'queen'
```

## Setup

In [ ]:
# !pip install sentence-transformers scikit-learn matplotlib
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded')

## Part 1 — Sentence similarity

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",
    "I enjoy eating pizza.",
    "Dogs love to play fetch.",
    "My dog chases the ball.",
]
embs = model.encode(sentences)
sim = cosine_similarity(embs)

print("Similarity matrix (rounded):")
for i, s in enumerate(sentences):
    row = "  ".join(f"{sim[i,j]:.2f}" for j in range(len(sentences)))
    print(f"  [{i}] {s[:30]:32s}  {row}")

## Part 2 — Semantic search

In [ ]:
corpus = [
    "Python is a high-level programming language.",
    "Machine learning models learn patterns from data.",
    "The French Revolution changed European history.",
    "Photosynthesis converts sunlight into glucose.",
    "Neural networks are inspired by the brain.",
    "Rome was not built in a day.",
    "Deep learning requires large datasets.",
    "Gradient descent minimises the loss function.",
]
corpus_embs = model.encode(corpus)

def search(query, top_k=3):
    q_emb = model.encode([query])
    scores = cosine_similarity(q_emb, corpus_embs)[0]
    top = scores.argsort()[::-1][:top_k]
    print(f"Query: {query}")
    for i in top:
        print(f"  [{scores[i]:.3f}] {corpus[i]}")
    print()

search("how do neural nets work?")
search("ancient history")
search("optimisation algorithms")

## Part 3 — Embedding dimensions & distance metrics

In [ ]:
import numpy as np

a = model.encode(["I love dogs"])
b = model.encode(["I adore puppies"])
c = model.encode(["The stock market fell today"])

def metrics(x, y, label):
    cos = cosine_similarity(x, y)[0][0]
    euc = np.linalg.norm(x - y)
    dot = np.dot(x[0], y[0])
    print(f"{label}: cosine={cos:.3f}  euclidean={euc:.3f}  dot={dot:.3f}")

metrics(a, b, "dogs vs puppies (similar)")
metrics(a, c, "dogs vs stocks (different)")

## Part 4 — Visualise embedding space

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

phrases = [
    "king", "queen", "man", "woman",
    "Paris", "France", "London", "England",
    "dog", "puppy", "cat", "kitten",
]
embs_2d = PCA(n_components=2).fit_transform(model.encode(phrases))

plt.figure(figsize=(8, 6))
for i, phrase in enumerate(phrases):
    x, y = embs_2d[i]
    plt.scatter(x, y, s=60)
    plt.annotate(phrase, (x, y), textcoords="offset points", xytext=(5, 3), fontsize=9)
plt.title("2D PCA of sentence embeddings")
plt.tight_layout()
plt.savefig("/tmp/embeddings_2d.png", dpi=120)
plt.show()
print("Plot saved")

## Summary

Embeddings underpin:
- **Semantic search** — find the most relevant document
- **Clustering** — group related content
- **Classification** — embed then feed into a classifier
- **RAG** — retrieve relevant chunks by similarity
- **De-duplication** — find near-duplicate content

Key insight: cosine similarity is preferred over Euclidean distance for text because it ignores magnitude and focuses purely on direction (meaning).